# 03 Vintages and cure
Observable vintage cells only; cure with competing risks and censoring.

In [1]:
import logging, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("streamlit").setLevel(logging.ERROR)
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.services import vintage, cure
c = vintage.quarterly_curves('ever_30p')
fig, ax = plt.subplots(figsize=(8, 4))
for q, g in list(c.groupby('VINTAGE_QTR'))[::4]:
    ax.plot(g.MOB, g.rate * 100, label=g.VINTAGE_LABEL.iloc[0])
ax.set_xlabel('Months on book'); ax.set_ylabel('Ever 30+ (%)'); ax.legend(fontsize=7)
plt.show()
print('curves end at the last observable MOB: max MOB by quarter ->', c.groupby('VINTAGE_LABEL').MOB.max().head().to_dict())

2026-10-04 20:32:36.035 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:36.038 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:36.131 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


curves end at the last observable MOB: max MOB by quarter -> {'M-12 to M-10': 10, 'M-15 to M-13': 13, 'M-18 to M-16': 16, 'M-21 to M-19': 19, 'M-24 to M-22': 22}


## Recent vs oldest vintages at fixed MOB checkpoints

In [3]:
pd.concat([vintage.comparison(m) for m in ['ever_30p', 'ever_60p']])[['METRIC', 'MOB', 'old_rate', 'recent_rate', 'diff', 'lo', 'hi', 'verdict']].round(4)

,METRIC,MOB,old_rate,recent_rate,diff,lo,hi,verdict
0,ever_30p,6,0.0859,0.0855,-0.0004,-0.0101,0.0091,no clear difference
4,ever_30p,12,0.1285,0.1202,-0.0083,-0.0194,0.0034,no clear difference
8,ever_30p,18,0.1395,0.1302,-0.0093,-0.0213,0.0026,no clear difference
12,ever_30p,24,0.1484,0.1343,-0.0141,-0.0269,-0.0020,lower
1,ever_60p,6,0.0127,0.0067,-0.0060,-0.0095,-0.0027,lower
5,ever_60p,12,0.0227,0.0118,-0.0109,-0.0157,-0.0063,lower
9,ever_60p,18,0.0261,0.0134,-0.0126,-0.0179,-0.0078,lower
13,ever_60p,24,0.0282,0.0160,-0.0122,-0.0179,-0.0072,lower


## Cure: primary (cumulative incidence) vs fixed-window estimates at 3 and 6 months

In [4]:
t = cure.cif_curves()
t[t.K.isin([3, 6])][['GRP', 'K', 'episodes', 'cif_cure', 'cif_cure_lo', 'cif_cure_hi', 'cif_co', 'fw_cure', 'fw_co']].round(4)

,GRP,K,episodes,cif_cure,cif_cure_lo,cif_cure_hi,cif_co,fw_cure,fw_co
2,All episodes,3,7575.0,0.8753,0.8658,0.8842,0.0144,0.9012,0.0142
3,All episodes,6,7575.0,0.9368,0.9294,0.9437,0.0341,0.9508,0.0287
6,Entry 30-59,3,7332.0,0.8889,0.8801,0.8972,0.0005,0.9139,0.0005
7,Entry 30-59,6,7332.0,0.9505,0.9441,0.9564,0.0204,0.9630,0.0163
10,Entry 60+,3,243.0,0.4473,0.3650,0.5310,0.4652,0.4703,0.4811
11,Entry 60+,6,243.0,0.4998,0.4117,0.5922,0.4768,0.5305,0.4573
14,From 1-29,3,7221.0,0.8818,0.8727,0.8901,0.0095,0.9053,0.0091
15,From 1-29,6,7221.0,0.9422,0.9354,0.9483,0.0290,0.9550,0.0241
18,From dormant,3,353.0,0.7273,0.6371,0.8178,0.1133,0.7885,0.1542
19,From dormant,6,353.0,0.8328,0.7435,0.9144,0.1378,0.8350,0.1553


In [5]:
chk = t.cif_cure + t.cif_co + t.still_30p
print('cure + charge-off + still 30+ sums to 1:', bool(((chk - 1).abs() < 1e-9).all()))
print(cure.redelinquency())

2026-10-04 20:32:36.225 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


cure + charge-off + still 30+ sums to 1: True
{'cured_with_window': 5252.0, 'redelinquent_6m': 1645.0, 'rate': 0.3132140137090632}
